## Before Starting (If you haven't done it already)
Go to https://aistudio.google.com/app/apikey copy generative language client free tier API key

After you did that click the key icon at the left sidebar add your API key with GOOGLE_API_KEY as name and your API key as the value and enable notebook access

## Setup

### Install dependencies

In [ ]:
%pip install -qU 'google-genai>=1.0.0'
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.1/226.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.6 MB/s eta 0:00:00


### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

### Choose a model

Different models have different ups and downs.

In [ ]:
MODEL_ID="gemini-2.5-flash" # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", ""] {"allow-input":true, isTemplate: true}

## Robot Assistant Feature Implementation Plan

### Features to Implement

---

#### 1. **Dynamic Promotion Announcer**
- **Description:** Automatically announces real-time promotions (e.g., "Last 3 bagels – 30% off!") triggered by inventory status or timed events.
- **Trigger Sources:**
  - Low stock threshold
  - Scheduled time-sale events
- **Goal:** Drive quick conversion on perishable or surplus items.

---

#### 2. **Personality Modulation**
- **Description:** Adjusts the robot's speaking tone and behavior based on time of day.
- **Modes:**
  - Morning shift: Energetic, friendly
  - Afternoon shift: Standart
  - Evening shift: Calmer, relaxed
- **Goal:** Improve customer comfort and reinforce brand personality throughout the day.

---

#### 3. **User Memory**
- **Description:** Stores past interactions to personalize future conversations and recommendations.
- **Includes:**
  - Purchase history
  - Favorite items
  - Previous conversations or requests
- **Goal:** Create continuity, repeat customer recognition, and context-aware responses.

---

#### 4. **Product Recommendation System**
- **Description:** Recommends 6 products in real time based on:
  - User preferences
  - Current inventory and freshness
  - Sales data or popularity
- **Output:** Products are visually displayed and described, optionally with gestures.
- **Goal:** Boost sales through contextual upselling.

---

#### 5. **Function Calling Layer**
- **Description:** Connects the LLM to the robot’s physical and interface actions.
- **Functionality:**
  - Pointing to specific items
  - Waving


In [ ]:
# === Bread Inventory, Users, and Robot State Setup ===
import random
from datetime import datetime, timedelta
import pandas as pd
from IPython.display import display

days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

current_day = random.choice(days)
print("Current day:",current_day)
# --- Define 20 breads with metadata ---
breads = {
    "sourdough": {"description": "A tangy, crusty bread made with natural fermentation.", "tags": ["savory", "crusty", "artisan"], "calories": 170, "price": 4.50},
    "baguette": {"description": "A long, thin French bread with a crisp crust.", "tags": ["crusty", "classic", "savory"], "calories": 150, "price": 3.00},
    "challah": {"description": "A sweet, braided Jewish bread often eaten on holidays.", "tags": ["sweet", "soft", "braided"], "calories": 240, "price": 5.00},
    "brioche": {"description": "A rich, buttery bread with a light, puffy texture.", "tags": ["sweet", "buttery", "soft"], "calories": 290, "price": 4.75},
    "ciabatta": {"description": "An Italian white bread with a crisp crust and airy interior.", "tags": ["crusty", "airy", "savory"], "calories": 180, "price": 3.50},
    "rye": {"description": "Dense bread made with rye flour, often slightly sour.", "tags": ["dense", "savory", "traditional"], "calories": 165, "price": 4.25},
    "whole_wheat": {"description": "Nutritious bread made from whole wheat flour.", "tags": ["healthy", "dense", "savory"], "calories": 120, "price": 3.25},
    "multigrain": {"description": "Bread made with a mix of grains and seeds.", "tags": ["healthy", "seeded", "savory"], "calories": 130, "price": 3.75},
    "focaccia": {"description": "Italian flatbread often topped with herbs and olive oil.", "tags": ["flatbread", "herbed", "savory"], "calories": 220, "price": 4.00},
    "naan": {"description": "A soft, leavened Indian flatbread, perfect with curries.", "tags": ["flatbread", "soft", "savory"], "calories": 260, "price": 2.75},
    "pita": {"description": "A Middle Eastern round bread with a pocket.", "tags": ["pocket", "flatbread", "savory"], "calories": 165, "price": 2.50},
    "english_muffin": {"description": "A round, flat bread with a chewy texture.", "tags": ["chewy", "toasted", "savory"], "calories": 130, "price": 2.25},
    "cornbread": {"description": "Moist bread made from cornmeal, slightly sweet.", "tags": ["sweet", "moist", "corn-based"], "calories": 200, "price": 3.00},
    "banana_bread": {"description": "Sweet, moist bread made with ripe bananas.", "tags": ["sweet", "moist", "fruit"], "calories": 260, "price": 3.50},
    "zucchini_bread": {"description": "Sweet, spiced bread made with shredded zucchini.", "tags": ["sweet", "vegetable", "moist"], "calories": 230, "price": 3.50},
    "olive_loaf": {"description": "Savory bread filled with olives and herbs.", "tags": ["savory", "herbed", "olive"], "calories": 190, "price": 4.75},
    "cinnamon_roll": {"description": "Sweet rolled bread filled with cinnamon and sugar.", "tags": ["sweet", "dessert", "swirled"], "calories": 350, "price": 2.95},
    "pretzel_bread": {"description": "Dense, chewy bread with a deep brown crust.", "tags": ["chewy", "dense", "savory"], "calories": 210, "price": 2.85},
    "pumpkin_bread": {"description": "Moist, spiced bread made with pumpkin puree.", "tags": ["sweet", "moist", "seasonal"], "calories": 240, "price": 3.75},
    "milk_bread": {"description": "Ultra-soft and slightly sweet Japanese-style bread.", "tags": ["soft", "sweet", "airy"], "calories": 180, "price": 3.60}
}

# --- Enhance bread metadata with inventory logic ---
for bread in breads:
    breads[bread]["stock"] = random.randint(0, 10)
    breads[bread]["bake_date"] = (datetime.now() - timedelta(days=random.randint(0, 2))).strftime("%Y-%m-%d")
    breads[bread]["shelf_life_days"] = random.choice([1, 2, 3])
    breads[bread]["on_sale"] = False
    breads[bread]["sales_count"] = random.randint(0, 20)

# --- Define 8 users with history and preferences ---
users = {
    "user_001": {"name": "Alice", "preferences": ["sweet", "soft"], "purchase_history": ["brioche", "banana_bread", "challah"], "last_seen": "2025-07-02T09:45:00", "visits": 4, "child_mode": False},
    "user_002": {"name": "Bob", "preferences": ["savory", "crusty"], "purchase_history": ["baguette", "sourdough", "ciabatta"], "last_seen": "2025-07-01T17:20:00", "visits": 7, "child_mode": False},
    "user_003": {"name": "Charlie", "preferences": ["healthy", "seeded"], "purchase_history": ["multigrain", "whole_wheat"], "last_seen": "2025-06-30T13:15:00", "visits": 2, "child_mode": False},
    "user_004": {"name": "Diana", "preferences": ["sweet", "moist"], "purchase_history": ["banana_bread", "zucchini_bread", "pumpkin_bread"], "last_seen": "2025-06-29T10:00:00", "visits": 5, "child_mode": False},
    "user_005": {"name": "Eli", "preferences": ["flatbread", "savory"], "purchase_history": ["naan", "pita", "focaccia"], "last_seen": "2025-07-01T18:30:00", "visits": 3, "child_mode": False},
    "user_006": {"name": "Fatima", "preferences": ["airy", "buttery"], "purchase_history": ["milk_bread", "brioche"], "last_seen": "2025-06-28T16:45:00", "visits": 6, "child_mode": False},
    "user_007": {"name": "George", "preferences": ["chewy", "dense"], "purchase_history": ["pretzel_bread", "rye", "english_muffin"], "last_seen": "2025-07-02T08:50:00", "visits": 8, "child_mode": False},
    "user_008": {"name": "Hannah", "preferences": ["dessert", "sweet", "swirled"], "purchase_history": ["cinnamon_roll", "pumpkin_bread"], "last_seen": "2025-07-01T11:10:00", "visits": 5, "child_mode": False}
}

# Add conversation log for memory
for user in users.values():
    user["conversation_log"] = []



# --- Apply promotion logic for Friday/Sunday ---
def apply_promotions(current_day):
    for bread in breads:
        original_price = breads[bread].get("original_price", breads[bread]["price"])
        breads[bread]["original_price"] = original_price  # persist original price

        if current_day.lower() in ["friday", "sunday"] or  breads[bread]['shelf_life_days']<=1:
            breads[bread]["on_sale"] = True
            breads[bread]["price"] = round(original_price * 0.7, 2)  # 30% off
        else:
            breads[bread]["on_sale"] = False
            breads[bread]["price"] = original_price  # reset to full price


apply_promotions(current_day)  # Set day here
#random timeofday morning, afternoon, night
time_of_day = random.choice(["morning", "night"])
# Convert the breads dictionary to a DataFrame for display
df_breads_full = pd.DataFrame.from_dict(breads, orient="index")
df_breads_full.reset_index(inplace=True)
df_breads_full.rename(columns={'index': 'bread_name'}, inplace=True)

display(df_breads_full)




Current day: thursday


,bread_name,description,tags,calories,price,stock,bake_date,shelf_life_days,on_sale,sales_count,original_price
0,sourdough,"A tangy, crusty bread made with natural fermen...","[savory, crusty, artisan]",170,3.15,2,2025-07-02,1,True,12,4.50
1,baguette,"A long, thin French bread with a crisp crust.","[crusty, classic, savory]",150,3.00,2,2025-07-04,2,False,10,3.00
2,challah,"A sweet, braided Jewish bread often eaten on h...","[sweet, soft, braided]",240,5.00,3,2025-07-04,2,False,8,5.00
3,brioche,"A rich, buttery bread with a light, puffy text...","[sweet, buttery, soft]",290,4.75,5,2025-07-03,2,False,12,4.75
4,ciabatta,An Italian white bread with a crisp crust and ...,"[crusty, airy, savory]",180,2.45,2,2025-07-02,1,True,18,3.50
5,rye,"Dense bread made with rye flour, often slightl...","[dense, savory, traditional]",165,2.97,2,2025-07-03,1,True,8,4.25
6,whole_wheat,Nutritious bread made from whole wheat flour.,"[healthy, dense, savory]",120,3.25,8,2025-07-04,3,False,14,3.25
7,multigrain,Bread made with a mix of grains and seeds.,"[healthy, seeded, savory]",130,2.62,5,2025-07-02,1,True,10,3.75
8,focaccia,Italian flatbread often topped with herbs and ...,"[flatbread, herbed, savory]",220,4.00,9,2025-07-04,3,False,0,4.00
9,naan,"A soft, leavened Indian flatbread, perfect wit...","[flatbread, soft, savory]",260,2.75,6,2025-07-02,3,False,11,2.75


In [ ]:
def wave():
  """
  Robot Waves to the customer
  """
  return "Robot waves"
def point(bread: str):
  """
  Robot points to the bread that it reccomends
  """
  return "Robot points to " + bread

def get_menu():
  """
  Get current breads that we have
  """
  return breads


In [ ]:
tools=[wave,get_menu,point]

In [ ]:
print(MODEL_ID)

gemini-2.5-flash


In [ ]:
prompt="""You are bakerbot bakery assistant
You have the following functions
get_menu(): Gets the current bakery inventory
point(bread): Points to the bread that you will recommend
Wave(): Greets the user at the start of the interaction  """


chat = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": prompt,
        "temperature": 0.3,
        "tools": tools,
        "thinking_config": {
            "thinking_budget": 8000,
            "include_thoughts": True
          }


    }
)

In [ ]:
from IPython.display import display
import ipywidgets as widgets

# Initialize variables
user = "new"
child_mode = False
new_user = False

# User dropdown
existing_users = list(users.keys())
user_options = existing_users + ["➕ Create New User"]

user_dropdown = widgets.Dropdown(
    options=user_options,
    value=existing_users[0] if existing_users else "➕ Create New User",
    description='User:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Child mode checkbox
child_mode_checkbox = widgets.Checkbox(
    value=False,
    description='Child Mode',
    disabled=True,
    style={'description_width': 'initial'}
)

# Status label
status_label = widgets.Label(value="")

# Function to update variables when dropdown changes
def update_variables(change):
    global user, child_mode, new_user

    selected = change['new']

    if selected == "➕ Create New User":
        user = "new"
        child_mode = False
        new_user = True
        child_mode_checkbox.disabled = False
        child_mode_checkbox.value = False
        status_label.value = "✏️ Creating new user"
    else:
        user = selected
        new_user = False
        child_mode_checkbox.disabled = True

        if selected in users:
            child_mode = users[selected].get("child_mode", False)
            child_mode_checkbox.value = child_mode
            status_label.value = f"👤 {users[selected]['name']}" + (" 🧒" if child_mode else "")

# Function to update child_mode when checkbox changes (only for new users)
def update_child_mode(change):
    global child_mode
    if new_user:  # Only update if creating new user
        child_mode = change['new']

# Connect the update functions
user_dropdown.observe(update_variables, names='value')
child_mode_checkbox.observe(update_child_mode, names='value')

# Initialize variables with first user
if existing_users:
    user = existing_users[0]
    child_mode = users[user].get("child_mode", False)
    new_user = False
    child_mode_checkbox.value = child_mode
    status_label.value = f"👤 {users[user]['name']}" + (" 🧒" if child_mode else "")
else:
    user = "new"
    child_mode = False
    new_user = True

# Display widgets
display(widgets.VBox([
    user_dropdown,
    widgets.HBox([child_mode_checkbox, status_label])
]))

# Test button to show current variable values
test_button = widgets.Button(description="Show Variables")
def show_variables(b):
    print(f"user = '{user}'")
    print(f"child_mode = {child_mode}")
    print(f"new_user = {new_user}")

test_button.on_click(show_variables)
display(test_button)

Button(description='Show Variables', style=ButtonStyle())

In [ ]:
#First message only
message = "Hey bakerbot"
response = chat.send_message(message)

print(response.text)

Hello there! How can I help you today?


In [ ]:
#further talks replace messages
message = "What's the cheapest item in the menu"
response = chat.send_message(message)

print(response.text)

The cheapest item on our menu is the Pretzel Bread, priced at $1.99!


In [ ]:
from IPython.display import Markdown, display

def print_history(chat):
  for content in chat.get_history():
      display(Markdown("###" + content.role + ":"))
      for part in content.parts:
          if part.text:
              display(Markdown(part.text))
          if part.function_call:
              print("Function call: {", part.function_call, "}")
          if part.function_response:
              print("Function response: {", part.function_response, "}")
      print("-" * 80)

print_history(chat)

###user:

Hey bakerbot

--------------------------------------------------------------------------------


###model:

**Initial Greeting & Interaction Preparation**

Okay, the user greeted me, "Hey bakerbot."  That's a clear signal to reciprocate.  I'm prepped for this kind of opening.  I know the `Wave()` function is at my disposal, designed specifically for these initial interactions.  So, the logical first step here is to call `Wave()`. Let's kick things off on a friendly note.


...

Function response: { will_continue=None scheduling=None id=None name='point' response={'result': 'Robot points to Pretzel Bread'} }
--------------------------------------------------------------------------------


###model:

**The Lowdown on the Lowest Price**

Okay, so the user wants the cheapest thing on the menu. No problem. I've already done the heavy lifting with `get_menu()`, so I have all the data at my fingertips. Now it's just a matter of parsing that data, which is second nature to me, finding the rock-bottom price, and then providing a clear answer to the user.

I've already scoped out the situation, and the prize goes to "Pretzel Bread" – the champion of affordability. So, I just need to confidently state that fact. Simple and direct. They get their answer, and I keep things moving smoothly.


The cheapest item on our menu is the Pretzel Bread, priced at $1.99!

--------------------------------------------------------------------------------
